# Fixing Links Notebook

Before running this notebook, make sure to run the following command in the terminal to install the required packages:

```bash
bundle install
bundle exec jekyll build
bundle exec htmlproofer _site > htmlproofer-output.txt 2>&1
ruby parse_htmlproofer_log.rb 
```

Each command should be run separately and the final two commands create files for all the htmlproofer errors and warnings. This notebook loads the final csv file to help you see what links exists. You will also need to install the `pandas` library if you haven't already. You can do this by running:

```bash
pip install pandas
```

## Load Libraries and Data

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("htmlproofer-report.csv")
# Lower case the column names
df.columns = df.columns.str.lower()
print(f"Number of errors: {len(df)}")

Number of errors: 4405


In [4]:
df.message.value_counts()

message
'a' tag is missing a reference                                                                                                      2190
External link https://github.com/programminghistorian/jekyll/commits/gh-pages failed (status code 429)                               491
internal image i/genericThumb.jpg does not exist                                                                                      10
External link https://twitter.com/jenniferisve failed (status code 400)                                                                9
External link https://twitter.com/rivaquiroga failed (status code 400)                                                                 9
                                                                                                                                    ... 
External link https://github.com/programminghistorian/jekyll/commits/gh-pages/en/project-team.md failed (status code 429)              1
External link https://github.com/

In [5]:
file_counts_df = df.file.value_counts().reset_index()
file_counts_df['count_index'] = file_counts_df.index

file_counts_df

,file,count,count_index
0,_site/es/lecciones/sitios-estaticos-con-jekyll...,85,0
1,_site/en/lessons/building-static-sites-with-je...,85,1
2,_site/assets/from-html-to-list-of-words-1/obo-...,52,2
3,_site/assets/normaliser-donnees-textuelles-pyt...,52,3
4,_site/fr/equipe-projet.html,33,4
...,...,...,...
498,_site/assets/mapping-with-python-leaflet/exerc...,2,498
499,_site/assets/mapping-with-python-leaflet/exerc...,2,499
500,_site/assets/mapping-with-python-leaflet/exerc...,2,500
501,_site/assets/sustainable-authorship-in-plain-t...,1,501


In [6]:
merged_df = df.merge(file_counts_df, on='file', how='outer').sort_values(by="count_index", ascending=True)

In [36]:
merged_df[merged_df.file.str.contains("_site/es/lecciones/sitios-estaticos-con-jekyll", na=False)].message.value_counts()

message
'a' tag is missing a reference                                                                                                                                       78
internally linking to #sectionwindows; the file exists, but the hash 'sectionwindows' does not                                                                        1
internally linking to #section1-9; the file exists, but the hash 'section1-9' does not                                                                                1
External link https://github.com/programminghistorian/jekyll/commits/gh-pages/es/lecciones/sitios-estaticos-con-jekyll-y-github-pages.md failed (status code 429)     1
External link https://github.com/programminghistorian/jekyll/commits/gh-pages failed (status code 429)                                                                1
External link https://jekyll-windows.juthilo.com/ failed with something very wrong.                                                                     

In [38]:
merged_df[merged_df.message == "'a' tag is missing a reference"].file.value_counts()

file
_site/en/lessons/building-static-sites-with-jekyll-github-pages.html                80
_site/es/lecciones/sitios-estaticos-con-jekyll-y-github-pages.html                  78
_site/pt/licoes/som-dados-sonificacao-historiadores.html                            17
_site/en/lessons/sonification.html                                                  17
_site/assets/from-html-to-list-of-words-1/obo-t17800628-33.html                     12
                                                                                    ..
_site/es/lecciones/introduccion-a-bash.html                                          4
_site/en/lessons/clustering-visualizing-word-embeddings.html                         4
_site/es/lecciones/introduccion-a-imageplot-y-la-visualizacion-de-metadatos.html     4
_site/es/lecciones/introduccion-a-markdown.html                                      4
_site/en/vacancies.html                                                              4
Name: count, Length: 493, dtype: int64

In [39]:
merged_df[merged_df.file.str.contains("_site/en/lessons/building-static-sites-with-jekyll-github-pages", na=False)]

,file,line,message,count,count_index
413,_site/en/lessons/building-static-sites-with-je...,533,'a' tag is missing a reference,85,1
412,_site/en/lessons/building-static-sites-with-je...,532,'a' tag is missing a reference,85,1
411,_site/en/lessons/building-static-sites-with-je...,531,'a' tag is missing a reference,85,1
410,_site/en/lessons/building-static-sites-with-je...,530,'a' tag is missing a reference,85,1
404,_site/en/lessons/building-static-sites-with-je...,520,'a' tag is missing a reference,85,1
...,...,...,...,...,...
435,_site/en/lessons/building-static-sites-with-je...,706,'a' tag is missing a reference,85,1
436,_site/en/lessons/building-static-sites-with-je...,716,'a' tag is missing a reference,85,1
437,_site/en/lessons/building-static-sites-with-je...,778,'a' tag is missing a reference,85,1
439,_site/en/lessons/building-static-sites-with-je...,823,'a' tag is missing a reference,85,1


In [30]:
import os
import re

EXTENSIONS = (".yml")

def replace_links_preserving_code_blocks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Match code blocks (triple backticks) and inline code (`...`)
    code_blocks = list(re.finditer(r"(```.*?```|`[^`]*`)", content, re.DOTALL))
    modified = content
    offset = 0

    for match in code_blocks:
        start, end = match.span()
        segment = content[start:end]

        # Temporarily mark this section to skip
        placeholder = f"%%CODEBLOCK{start}%%"
        modified = modified[:start + offset] + placeholder + modified[end + offset:]
        offset += len(placeholder) - (end - start)

    # Replace all http:// with https://
    modified = re.sub(r"http://", "https://", modified)

    # Restore code blocks untouched
    for match in code_blocks:
        start = match.start()
        placeholder = f"%%CODEBLOCK{start}%%"
        modified = modified.replace(placeholder, match.group(0))

    if content != modified:
        print(f"✅ Updated: {file_path}")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(modified)

def process_all_files(root="."):
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.endswith(EXTENSIONS) and "ph_authors" in fname:
                replace_links_preserving_code_blocks(os.path.join(dirpath, fname))

process_all_files()

✅ Updated: ./_data/ph_authors.yml
